In [ ]:
import masknmf
import fastplotlib as fpl
import tifffile
import torch
import numpy as np
%load_ext autoreload




In [ ]:
# Point to your dataset
#data = tifffile.imread("/mnt/data1/vps2116/Widefield_MtlabOutputs/cm9999/cm9999_9 Registration.npz") # Shape (frames, height, width)
import numpy as np

# Load the .npz archive
data = np.load("/mnt/data1/vps2116/Widefield_MtlabOutputs/cm9999/cm9999_9_Registration/run9E.npz")


# Access the array
video = data['moco']  # (frames, height, width)

# Get the mean image (or any other image you want)
mean_img = torch.from_numpy(np.mean(video, axis = 0)) #(Shape (height, width))
#mean_img = torch.tensor(np.mean(video, axis=0), dtype=torch.float32)



'''
Define the transformation as a function that takes (frames, height, width) torch.tensor as input
and outputs a tensor of the same shape
'''
mean_sub = lambda x:x - mean_img[None, :, :]

# Define an array object that describes this transformed dataset
mean_sub_array = masknmf.FilteredArray(video, 
                                       mean_sub,
                                      device='cpu')

In [ ]:
# Visualize this in imagewidget. 
# Rendering speed depends on whether computations are performed on cpu/gpu and on whether the dataset is in RAM or not
iw = fpl.ImageWidget(data=[mean_sub_array, video, mean_img.numpy()],
                     names=None,
                     figure_shape=(1, 3))

iw.cmap = "gray"
iw.show()

In [ ]:
#Get the mean image to import into draw_mask.py to make mask
import matplotlib.pyplot as plt

plt.imsave("mean_image_for_mask_18.png", mean_img.numpy(), cmap='gray') #Now locally load the mean with brain_mask.py
mean_img.shape


In [ ]:
#After using draw_mask.py, load in binary brain mask and apply to data
import numpy as np
import torch

# Load the brain mask
brain_mask = torch.tensor(np.load("/mnt/data1/vps2116/Widefield_MtlabOutputs/cm9999/cm9999_9_Registration/resid_ac_zero_blood_mask_9.npy")).float()

# Apply it to the video
mask_brain = lambda x: x * brain_mask
masked_array = masknmf.FilteredArray(video, mask_brain, device='cpu')

# Evaluate to NumPy array (no .cpu() needed!)
masked_np = masked_array[:]

# Save to .npz
np.savez_compressed("/mnt/data1/vps2116/Widefield_MtlabOutputs/cm9999/cm9999_9_Registration/run9E_masked_blood.npz", moco=masked_np)
print("✅ Saved masked video to masked_video.npz with key 'moco'")




In [ ]:
#Add the horizontal and vertical shifts from original moco data to masked data
import numpy as np

# Load original file that contains 'shifts'
original_npz = np.load("/mnt/data1/vps2116/Widefield_MtlabOutputs/cm9999/cm9999_9_Registration/run9E.npz")  # adjust name as needed
shifts = original_npz['shifts']

# Load masked video that only has 'moco'
masked_npz = np.load("/mnt/data1/vps2116/Widefield_MtlabOutputs/cm9999/cm9999_9_Registration/run9E_masked_blood.npz")
moco = masked_npz['moco']

# Save both into a new file (or overwrite)
np.savez_compressed(
    "/mnt/data1/vps2116/Widefield_MtlabOutputs/cm9999/cm9999_9_Registration/run9E_masked_shifts_blood.npz",
    moco=moco,
    shifts=shifts
)

print("✅ Masked video with 'shifts' saved.")


In [ ]:
# Point to your dataset
#data = tifffile.imread("/mnt/data1/vps2116/Widefield_MtlabOutputs/cm9999/cm9999_9 Registration.npz") # Shape (frames, height, width)
import numpy as np

# Load the .npz archive
data = np.load("/mnt/data1/vps2116/Widefield_MtlabOutputs/cm9999/cm9999_9_Registration/run9E_masked.npz")


# Access the array
video_masked = data['moco']  # (frames, height, width)

# Get the mean image (or any other image you want)
mean_img = torch.from_numpy(np.mean(video, axis = 0)) #(Shape (height, width))
#mean_img = torch.tensor(np.mean(video, axis=0), dtype=torch.float32)



'''
Define the transformation as a function that takes (frames, height, width) torch.tensor as input
and outputs a tensor of the same shape
'''
mean_sub = lambda x:x - mean_img[None, :, :]

# Define an array object that describes this transformed dataset
mean_sub_array = masknmf.FilteredArray(video, 
                                       mean_sub,
                                      device='cpu')

In [ ]:
# Visualize this in imagewidget. 
# Rendering speed depends on whether computations are performed on cpu/gpu and on whether the dataset is in RAM or not
iw = fpl.ImageWidget(data=[mean_sub_array, video_masked, mean_img.numpy()],
                     names=None,
                     figure_shape=(1, 3))

iw.cmap = "gray"
iw.show()